# Identifier fragmentation analysis

**Purpose:** Present committed token fragmentation and vocabulary-utilization summaries for tokenizer-efficiency mechanisms.

**Inputs (committed result/provenance files read):**
- [`results/tables/fragmentation.csv`](../results/tables/fragmentation.csv)
- [`results/tables/vocab_utilization.csv`](../results/tables/vocab_utilization.csv)
- [`results/tables/fig05_content.csv`](../results/tables/fig05_content.csv)

**Outputs:** Rendered fragmentation/utilization tables in this notebook only; no files are written.

**Expected runtime:** ~3 seconds (loads committed CSV/JSON/parquet files only).

**Traceability links:** [`FINAL_REPORT.md`](../results/reports/FINAL_REPORT.md) and [`CLAIM_LEDGER.md`](../publication/CLAIM_LEDGER.md).

**Execution notes:** Analysis-only notebook. It resolves `ROOT` by walking upward to `plan.md`, uses no absolute paths or machine identifiers, does not download corpora, does not retrain tokenizers, and has no hidden state. Randomness: none used. Exploratory analysis: none; cells below directly present committed result files.


In [1]:
import json
from pathlib import Path

import pandas as pd
from IPython.display import display

ROOT = Path.cwd()
while not (ROOT / 'plan.md').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent

def rel(path):
    path = Path(path)
    try:
        return str(path.relative_to(ROOT))
    except ValueError:
        return str(path)

def note_missing(path):
    print(f'NOTE: optional input is missing or empty: {path}. The notebook continues without it.')

def load_csv(path):
    p = ROOT / path
    if not p.exists():
        note_missing(path)
        return pd.DataFrame()
    return pd.read_csv(p)

def load_parquet(path):
    p = ROOT / path
    if not p.exists():
        note_missing(path)
        return pd.DataFrame()
    try:
        return pd.read_parquet(p)
    except Exception as exc:
        print(f'NOTE: could not read optional parquet {path}: {exc}')
        return pd.DataFrame()

def load_json(path):
    p = ROOT / path
    if not p.exists():
        note_missing(path)
        return {}
    return json.loads(p.read_text())

def load_text(path):
    p = ROOT / path
    if not p.exists():
        note_missing(path)
        return ''
    return p.read_text()

def safe_read_json_path(path):
    try:
        return json.loads(Path(path).read_text())
    except Exception as exc:
        print(f'NOTE: could not read {rel(path)}: {exc}')
        return {}

print(f'Repository root: {rel(ROOT)}')


Repository root: .


## Fragmentation mechanism

Authoritative files: [`fragmentation.csv`](../results/tables/fragmentation.csv), [`vocab_utilization.csv`](../results/tables/vocab_utilization.csv), and [`fig05_content.csv`](../results/tables/fig05_content.csv).

In [2]:
fragmentation = load_csv('results/tables/fragmentation.csv')
vocab = load_csv('results/tables/vocab_utilization.csv')
content = load_csv('results/tables/fig05_content.csv')

In [3]:
if not fragmentation.empty:
    pivot = fragmentation.pivot(index='category', columns='tokenizer_id', values='mean_tokens')
    display(pivot.sort_index().round(3))
else:
    note_missing('results/tables/fragmentation.csv')

tokenizer_id,azure-bytebpe-16384,general-bytebpe-16384,mixed-bytebpe-16384
category,,,
cli_flag,4.561,8.016,4.855
identifier_camel,2.883,6.521,3.449
identifier_snake,4.913,7.395,5.308
prose_word,1.636,2.882,1.921
resource_provider,10.547,22.640,11.864
url_path,26.354,42.869,27.663


In [4]:
if not vocab.empty:
    display(vocab.sort_values('tokenizer_id'))
else:
    note_missing('results/tables/vocab_utilization.csv')

,tokenizer_id,vocab_size,frac_used,used_ge_10,used_ge_100,used_ge_1000,unused,top1pct_share,top10pct_share,mean_token_byte_len
1,azure-bytebpe-16384,16384,0.8174,9321,2659,362,2992,0.5322,0.8436,6.356
0,general-bytebpe-16384,16384,0.4848,4999,2176,539,8441,0.6576,0.9528,6.171
2,mixed-bytebpe-16384,16384,0.6803,8066,2846,395,5238,0.5227,0.8521,5.956


In [5]:
if not content.empty:
    display(content)
else:
    note_missing('results/tables/fig05_content.csv')

,content_type,reduction_pct,ci_low,ci_high,n
0,prose,36.747816,35.865836,37.628845,900
1,heading,37.494553,30.664228,45.917734,44
2,table,44.297973,34.843493,47.253742,6
3,inline_code,33.283686,28.251609,42.914635,14
